# 04 — Hyperparameter Tuning (Optuna)

This notebook runs Optuna to tune a tree model (XGBoost and/or LightGBM) for the `Crime Solved` target.

Notes:
- Tuning can be slow on the full dataset. There is a `SAMPLE_N` knob below to tune on a subset first.
- The tuned pipeline is saved under `models/`.


In [ ]:
# Optuna tuning
from pathlib import Path
import sys

import numpy as np
import optuna

from sklearn.model_selection import StratifiedKFold, cross_val_score

_cwd = Path.cwd()
PROJECT_ROOT = _cwd if (_cwd / "data").exists() else _cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import DataPaths, load_processed_split
from src.train import TrainConfig, dataframe_X_y, fit_pipeline, make_pipeline, save_model
from src.evaluate import evaluate_binary

paths = DataPaths.from_project_root(PROJECT_ROOT)

train_df = load_processed_split(paths.processed_dir, "train")
val_df = load_processed_split(paths.processed_dir, "val")
test_df = load_processed_split(paths.processed_dir, "test")

X_train, y_train = dataframe_X_y(train_df)
X_val, y_val = dataframe_X_y(val_df)
X_test, y_test = dataframe_X_y(test_df)

# Optional: tune on a subset first for speed
SAMPLE_N = 120_000
if len(X_train) > SAMPLE_N:
    idx = X_train.sample(n=SAMPLE_N, random_state=42).index
    X_tune = X_train.loc[idx].reset_index(drop=True)
    y_tune = y_train.loc[idx].reset_index(drop=True)
else:
    X_tune, y_tune = X_train, y_train

cfg = TrainConfig(random_state=42, n_jobs=-1)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)


def tune_model(model_name: str, n_trials: int = 25):
    def objective(trial: optuna.Trial):
        if model_name == "xgb":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 200, 800),
                "max_depth": trial.suggest_int("max_depth", 3, 10),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            }
        elif model_name == "lgb":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "num_leaves": trial.suggest_int("num_leaves", 31, 255),
                "max_depth": trial.suggest_int("max_depth", -1, 12),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "min_child_samples": trial.suggest_int("min_child_samples", 10, 200),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            }
        else:
            raise ValueError("model_name must be 'xgb' or 'lgb'")

        pipe = make_pipeline(X_tune, model_name, model_params=params, cfg=cfg)
        scores = cross_val_score(pipe, X_tune, y_tune, scoring="roc_auc", cv=cv, n_jobs=1)
        return float(np.mean(scores))

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    return study


# Run tuning (adjust trials as needed)
study_xgb = tune_model("xgb", n_trials=20)
print("Best XGB AUC:", study_xgb.best_value)
print("Best XGB params:", study_xgb.best_params)

study_lgb = tune_model("lgb", n_trials=20)
print("Best LGB AUC:", study_lgb.best_value)
print("Best LGB params:", study_lgb.best_params)

# Fit best models on full training data and compare on validation
xgb_best = fit_pipeline(make_pipeline(X_train, "xgb", model_params=study_xgb.best_params, cfg=cfg), X_train, y_train)
lgb_best = fit_pipeline(make_pipeline(X_train, "lgb", model_params=study_lgb.best_params, cfg=cfg), X_train, y_train)

xgb_val = evaluate_binary(xgb_best, X_val, y_val).roc_auc
lgb_val = evaluate_binary(lgb_best, X_val, y_val).roc_auc
print("Val ROC-AUC | XGB:", round(xgb_val, 4), "| LGB:", round(lgb_val, 4))

best_name = "xgb" if xgb_val >= lgb_val else "lgb"
best_pipe = xgb_best if best_name == "xgb" else lgb_best
print("Selected:", best_name)

# Final test evaluation
from src.evaluate import plot_confusion, plot_roc_curve

test_res = evaluate_binary(best_pipe, X_test, y_test)
print("Test ROC-AUC:", round(test_res.roc_auc, 4))
plot_confusion(test_res.confusion, title=f"Confusion matrix — tuned {best_name} (test)")
plot_roc_curve(best_pipe, X_test, y_test, title=f"ROC — tuned {best_name} (test)")

# Save tuned pipeline(s)
save_model(xgb_best, PROJECT_ROOT / "models" / "xgb_best.joblib")
save_model(lgb_best, PROJECT_ROOT / "models" / "lgb_best.joblib")
print("Saved tuned models to models/")